In [1]:
import fitz               # PyMuPDF
import pandas as pd
from pathlib import Path

In [2]:
PDF_PATH   = Path("packing_slips.pdf")
CSV_PATH   = Path("po_order.csv")
OUTPUT_PDF = Path("packing_slips_reordered.pdf")

In [12]:
src_doc = fitz.open(PDF_PATH)

In [21]:
# extract PO numbers page‑by‑page
records = []
for page_no, page in enumerate(src_doc, start=1):           # 1‑based page numbers
    lines = page.get_text().splitlines()
    print(lines)
    try:
        idx = lines.index("ORDER#:")          # <— case‑sensitive; adjust if needed
        po = lines[idx + 2].strip()              # “3 lines below” rule
    except (ValueError, IndexError):
        po = None                                # couldn’t find the tag
    records.append({"page_no": page_no, "po": po})
page_df = pd.DataFrame(records)

['Questions?', 'www.kohls.com/order', 'RECEIPT ID#:', 'ORDER#:', '999-9281-7488-7097-9126-3788-4920', '6647302609_1', 'Returns', 'ship to:', "Kohl's Return Center", '9998 All Points Parkway', 'Plainfield, IN 46168', 'ra:', '6647302609_1', 'Ship to:', 'Barbara Sangeado', '13 clinton ave', 'Newport, RI 02840', 'US', 'Email: barbaras42@juno.com', 'Sold to:', 'Barbara Sangeado', '13 clinton ave', 'Newport, RI 02840', 'US', 'Thank you for your order!', 'We hope you enjoy your purchase from Kohls.com.', 'Use this form to return your online purchase(s) by', 'mail or at any one of our stores nationwide.', 'Premium electronics and warranty products may', 'not be returned by mail and are subject to a', 'modified return policy.', 'SKU#', 'UPC#', 'DESCRIPTION', 'QTY.', 'ORD.', 'QTY.', 'SENT', 'UNIT', 'COST', 'RETURN', 'QTY.', 'RETURN', 'CODE', '94694160', '883954015944', 'JSB300641-7.5-BLK Bracelet ', 'BLACK 7.5"', '1', '1', '$950.00', 'T1', 'SHIPPING METHOD', 'USPS First-Class Mail', 'Transaction

In [22]:
# 2.  Read the desired PO sequence from the CSV  -------------------
order_df = pd.read_csv(CSV_PATH)                 # assume a column called po
order_df["po"] = order_df["po"].astype(str).str.strip()

In [23]:
dest_doc      = fitz.open()                      # empty document
matched_pages = set()     

In [24]:
# helper: fast mapping PO → list of source page indices (0‑based)
po_to_pages = (
    page_df
    .dropna(subset=["po"])       # ignore pages where PO wasn’t detected
    .groupby("po")["page_no"]
    .apply(lambda s: [p-1 for p in s])           # convert to 0‑based
    .to_dict()
)

In [25]:
# 3a. pages whose PO appears in the CSV, in the CSV order
for po in order_df["po"]:
    pages = po_to_pages.get(po)
    if not pages:                      # PO was in CSV but not in the PDF
        print(f"⚠️  PO {po} not found in PDF — skipped")
        continue
    for pg in pages:
        dest_doc.insert_pdf(src_doc, from_page=pg, to_page=pg)
        matched_pages.add(pg)

In [ ]:
# 3b. pages whose PO is **not** listed in the CSV
for pg in range(len(src_doc)):
    if pg not in matched_pages:
        dest_doc.insert_pdf(src_doc, from_page=pg, to_page=pg)


In [10]:
dest_doc.save(OUTPUT_PDF)
dest_doc.close()
src_doc.close()

In [11]:
print(f"Done!  → {OUTPUT_PDF.resolve()}")

Done!  → /home/ymajan/Documents/Sorting Slips/src/packing_slips_reordered.pdf
